# Axe 2 : le sentiment de stagnation — exploration

**Question pour la DRH :** qui se sent stagner, à quel moment de la carrière, et pourquoi ?

Ce notebook sert à explorer les données avant d'écrire `analyse/axe_2.py`.
Étape 1 : prendre en main la colonne `JobSatComplacent`, ses réponses et le nombre de personnes qui y ont répondu.

## 1. Charger la population commune

In [1]:
import sys

# le notebook est dans notebooks/ : on ajoute la racine du projet pour importer analyse/
sys.path.append("..")

import pandas as pd
from analyse.chargement import charger
from analyse.population import preparer

df = preparer(charger(2026))
df.shape

(22839, 216)

Zone par défaut du projet : France et pays frontaliers ensemble (voir `ORGANISATION.md`).
La France seule a trop peu de répondants dès qu'on ajoute un filtre.

In [2]:
ZONE_DEFAUT = ["France", "Pays frontaliers"]

zone = df[df["Zone"].isin(ZONE_DEFAUT)]
print("Personnes en France et pays frontaliers :", len(zone))

Personnes en France et pays frontaliers : 3376


## 2. La colonne `JobSatComplacent` et ses réponses

In [3]:
zone["JobSatComplacent"].value_counts(dropna=False)

JobSatComplacent
NaN                                                        1768
I do not feel complacent                                    624
Burnout/tech fatigue                                        292
Economic uncertainty making me hesitant to switch roles     262
AI automating the challenging parts of my job               172
Lack of upward mobility due to corporate tightening         138
Other, please specify                                       120
Name: count, dtype: int64

In [4]:
# une seule réponse par personne, ou plusieurs séparées par « ; » ?
print("Réponses multiples :", zone["JobSatComplacent"].dropna().str.contains(";").sum())

Réponses multiples : 0


In [5]:
# traduction pour l'affichage (consignes/saad_axe-2.md)
TRADUCTION = {
    "I do not feel complacent": "Pas de sentiment de stagnation",
    "Burnout/tech fatigue": "Épuisement",
    "Economic uncertainty making me hesitant to switch roles": "Incertitude économique",
    "AI automating the challenging parts of my job": "L'IA fait les tâches intéressantes",
    "Lack of upward mobility due to corporate tightening": "Pas de perspective d'évolution",
    "Other, please specify": "Autre",
}

reponses = zone["JobSatComplacent"].dropna().map(TRADUCTION)
tableau = pd.DataFrame({
    "repondants": reponses.value_counts(),
    "part": reponses.value_counts(normalize=True).round(3),
})
tableau

,repondants,part
JobSatComplacent,,
Pas de sentiment de stagnation,624,0.388
Épuisement,292,0.182
Incertitude économique,262,0.163
L'IA fait les tâches intéressantes,172,0.107
Pas de perspective d'évolution,138,0.086
Autre,120,0.075


## 3. Qui a répondu à la question ?

In [6]:
def taux_reponse(tableau):
    # nombre de personnes, nombre de réponses à la question, et part de réponses
    return pd.Series({
        "personnes": len(tableau),
        "repondants": tableau["JobSatComplacent"].notna().sum(),
        "taux": round(tableau["JobSatComplacent"].notna().mean(), 3),
    })

pd.DataFrame({
    "France et pays frontaliers": taux_reponse(zone),
    "France seule": taux_reponse(df[df["Zone"] == "France"]),
    "Toute la population": taux_reponse(df),
}).T.astype({"personnes": int, "repondants": int})

,personnes,repondants,taux
France et pays frontaliers,3376,1608,0.476
France seule,612,284,0.464
Toute la population,22839,7540,0.330


In [7]:
# le taux de réponse change-t-il avec l'expérience ?
(zone.groupby("Tranche_experience", observed=True)
     .apply(taux_reponse)
     .astype({"personnes": int, "repondants": int}))

,personnes,repondants,taux
Tranche_experience,,,
0-1,92,51,0.554
2-3,249,119,0.478
4-6,446,227,0.509
7-10,617,295,0.478
11-20,1111,536,0.482
21+,832,377,0.453


In [8]:
# ceux qui ont répondu ont-ils aussi donné leur satisfaction (JobSat) ?
pd.crosstab(zone["JobSat"].notna(), zone["JobSatComplacent"].notna(),
            rownames=["JobSat renseigné"], colnames=["A répondu à JobSatComplacent"])

A répondu à JobSatComplacent,False,True
JobSat renseigné,,
False,20,0
True,1748,1608


## 4. Je remarque / je me demande

**Je remarque**

- Seules **1 608 personnes sur 3 376** ont répondu en France et pays frontaliers, soit **47,6 %**. Sur toute la population, c'est **33,0 %**. Les consignes parlent de « 45 % » : c'est un arrondi, les vrais chiffres sont ceux-là.
- **Une seule réponse par personne** : pas de « ; ». Les parts des réponses font donc 100 % à elles toutes.
- Parmi ceux qui ont répondu, **38,8 %** ne se sentent pas en stagnation. Donc **61,2 %** citent une cause.
- La première cause est **l'épuisement (18,2 %)**, puis **l'incertitude économique (16,3 %)**. Le manque de perspective d'évolution n'arrive qu'en 5ᵉ position (8,6 %), alors que le message de départ du projet parle surtout d'évolution : à vérifier dans la suite.
- Le taux de réponse varie peu avec l'expérience, entre 45 % et 55 % : pas de tranche qui répond beaucoup plus que les autres.
- La tranche **0-1 an** n'a que 92 personnes et **51 répondants**, juste au-dessus du seuil de 50. Avec un filtre de secteur, elle passera sous le seuil.
- En **France seule**, il n'y a que **284 répondants** : la zone par défaut France et pays frontaliers est nécessaire pour cet axe.
- Tous ceux qui ont répondu ont aussi donné leur satisfaction (`JobSat`) : on pourra croiser les deux.

**Je me demande**

- Pourquoi plus de la moitié n'ont pas répondu ? Si ceux qui se sentent bloqués ont plus répondu, la part de 61,2 % est trop haute. Il faut l'écrire sur la page.
- La part en stagnation monte-t-elle avec l'expérience, et à partir de quelle tranche ? (prochaine étape)
- Les causes sont-elles différentes en ESN et chez les éditeurs de logiciels ? (prochaine étape)
- « Complacent » veut plutôt dire « s'installer, ne plus chercher à bouger ». Le mot « stagnation » choisi par le groupe est-il le bon ? À discuter avec le groupe.